In [5]:
import pandas as pd

ETHOGRAM_FILE = "/home/feliciano/Downloads/00118_Acoustique_Poisson_2026-07-27.xlsx"
OUTPUT_CSV = "ethogram_intervals.csv"

# =====================================================
# LOAD
# =====================================================

df = pd.read_excel(
    ETHOGRAM_FILE,
    engine="openpyxl"
)

df.columns = [str(c).strip() for c in df.columns]

# =====================================================
# DATETIME
# =====================================================

df = df.dropna(subset=["Date"])

df["start_dt"] = pd.to_datetime(
    df["Date"].astype(str)
    + " "
    + df["Time entered"].astype(str),
    errors="coerce"
)

df["end_dt"] = pd.to_datetime(
    df["Date"].astype(str)
    + " "
    + df["Exit time"].astype(str),
    errors="coerce"
)

# =====================================================
# TRUE FEEDING DETECTOR
# =====================================================

def is_feeding_row(row):

    fields = []

    for col in df.columns:

        value = row.get(col)

        if pd.notna(value):

            fields.append(
                str(value).strip().upper()
            )

    text = " | ".join(fields)

    feeding_terms = [
        "ALIMENTATION",
        "NOURRISSAGE",
        "NOURRISSAGE AUTOMATIQUE",
        "ALIMENTATION AUTOMATIQUE",
        "NOURRISSAGE DES BASSINS"
    ]

    # must match a standalone feeding event
    for term in feeding_terms:

        if term == text:
            return True

        if f"| {term} |" in text:
            return True

        if text.endswith(term):
            return True

    return False


# =====================================================
# FIND FEEDING EVENTS
# =====================================================

feeding_idx = []

for idx, row in df.iterrows():

    if is_feeding_row(row):

        feeding_idx.append(idx)

print("Feeding rows:", len(feeding_idx))

# =====================================================
# BUILD INTERVALS
# =====================================================

intervals = []

for idx in feeding_idx:

    feed = df.loc[idx]

    feed_start = feed["start_dt"]
    feed_end = feed["end_dt"]

    if pd.isna(feed_start) or pd.isna(feed_end):
        continue

    # ---------------------------
    # previous observation
    # ---------------------------

    prev_idx = idx - 1

    while prev_idx >= 0:

        prev = df.loc[prev_idx]

        if (
            not is_feeding_row(prev)
            and pd.notna(prev["start_dt"])
            and pd.notna(prev["end_dt"])
        ):

            intervals.append(
                {
                    "start": prev["start_dt"],
                    "end": prev["end_dt"],
                    "class": "prefeeding"
                }
            )

            break

        prev_idx -= 1

    # ---------------------------
    # feeding
    # ---------------------------

    intervals.append(
        {
            "start": feed_start,
            "end": feed_end,
            "class": "feeding"
        }
    )

    # ---------------------------
    # next observation
    # ---------------------------

    next_idx = idx + 1

    while next_idx < len(df):

        nxt = df.loc[next_idx]

        if (
            not is_feeding_row(nxt)
            and pd.notna(nxt["start_dt"])
            and pd.notna(nxt["end_dt"])
        ):

            intervals.append(
                {
                    "start": nxt["start_dt"],
                    "end": nxt["end_dt"],
                    "class": "postfeeding"
                }
            )

            break

        next_idx += 1


# =====================================================
# SAVE
# =====================================================

intervals_df = pd.DataFrame(intervals)

if len(intervals_df) == 0:

    raise RuntimeError(
        "No feeding intervals found."
    )

intervals_df = (
    intervals_df
    .drop_duplicates()
    .sort_values("start")
    .reset_index(drop=True)
)

intervals_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print()
print(intervals_df["class"].value_counts())
print()
print(intervals_df.head(20))
print()
print("Saved:", OUTPUT_CSV)

Feeding rows: 23

class
prefeeding     20
feeding        20
postfeeding    20
Name: count, dtype: int64

                 start                 end        class
0  2026-04-23 12:00:00 2026-04-23 12:10:00   prefeeding
1  2026-04-23 12:20:00 2026-04-23 12:25:00      feeding
2  2026-04-23 12:25:00 2026-04-23 12:35:00  postfeeding
3  2026-04-23 13:50:00 2026-04-23 14:00:00   prefeeding
4  2026-04-23 14:00:00 2026-04-23 14:05:00      feeding
5  2026-04-23 14:05:00 2026-04-23 14:15:00  postfeeding
6  2026-04-26 09:50:00 2026-04-26 10:00:00   prefeeding
7  2026-04-26 10:00:00 2026-04-26 10:05:00      feeding
8  2026-04-26 10:05:00 2026-04-26 10:15:00  postfeeding
9  2026-04-26 11:30:00 2026-04-26 11:40:00   prefeeding
10 2026-04-26 11:40:00 2026-04-26 11:45:00      feeding
11 2026-04-26 11:45:00 2026-04-26 11:55:00  postfeeding
12 2026-04-26 13:10:00 2026-04-26 13:20:00   prefeeding
13 2026-04-26 13:20:00 2026-04-26 13:25:00      feeding
14 2026-04-26 13:25:00 2026-04-26 13:35:00  postfeeding

In [6]:
import pandas as pd

df = pd.read_csv("ethogram_intervals.csv")

print(df.head(20))
print()
print(df["class"].value_counts())

                  start                  end        class
0   2026-04-23 12:00:00  2026-04-23 12:10:00   prefeeding
1   2026-04-23 12:20:00  2026-04-23 12:25:00      feeding
2   2026-04-23 12:25:00  2026-04-23 12:35:00  postfeeding
3   2026-04-23 13:50:00  2026-04-23 14:00:00   prefeeding
4   2026-04-23 14:00:00  2026-04-23 14:05:00      feeding
5   2026-04-23 14:05:00  2026-04-23 14:15:00  postfeeding
6   2026-04-26 09:50:00  2026-04-26 10:00:00   prefeeding
7   2026-04-26 10:00:00  2026-04-26 10:05:00      feeding
8   2026-04-26 10:05:00  2026-04-26 10:15:00  postfeeding
9   2026-04-26 11:30:00  2026-04-26 11:40:00   prefeeding
10  2026-04-26 11:40:00  2026-04-26 11:45:00      feeding
11  2026-04-26 11:45:00  2026-04-26 11:55:00  postfeeding
12  2026-04-26 13:10:00  2026-04-26 13:20:00   prefeeding
13  2026-04-26 13:20:00  2026-04-26 13:25:00      feeding
14  2026-04-26 13:25:00  2026-04-26 13:35:00  postfeeding
15  2026-05-01 09:30:00  2026-05-01 09:40:00   prefeeding
16  2026-05-01

In [4]:
import pandas as pd

file = "/home/feliciano/Downloads/00118_Acoustique_Poisson_2026-07-27.xlsx"

df = pd.read_excel(
    file,
    engine="openpyxl"
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 500)

for col in df.columns:

    mask = (
        df[col]
        .astype(str)
        .str.contains(
            "ALIMENT|NOURR",
            case=False,
            na=False
        )
    )

    if mask.any():

        print("\n" + "="*80)
        print("COLUMN:", col)
        print("="*80)

        print(
            df.loc[mask]
              .head(20)
              .to_string()
        )


COLUMN: Notes
         Date Time entered Exit time  Duration    BASSINS Code  Behavior / Activity                          Description                                            Notes  Unnamed: 9 Unnamed: 10          Unnamed: 11                   Unnamed: 12
6  2026-04-23          NaN       NaN  00:00:00        NaN  NaN                  NaN                                  NaN     ALIMENT BEAUCOUP EN SURFACE TOUS LES BASSINS         NaN          P6  Reactivity to noise  Reaction to a tool or impact
7  2026-04-23     10:45:00  10:55:00  00:10:00        B-1   P5                  NaN                                  NaN                  AGITATION ET BONNE ALIMENTATION         NaN         NaN                  NaN                           NaN
9  2026-04-23     10:45:00  10:55:00  00:10:00        B-2   P5                  NaN                                  NaN               BONNE ALIMENTATION SAUT /AGITATION         NaN         NaN                  NaN                           NaN
11 20

In [7]:
import os
import shutil
import pandas as pd

SEGMENTS_DIR = "/media/feliciano/Aux/AI_AFS_DATASET/segmented_2s"
OUTPUT_DIR = "/media/feliciano/Aux/AI_AFS_DATASET/classified"

intervals = pd.read_csv(
    "ethogram_intervals.csv",
    parse_dates=["start", "end"]
)

for cls in [
    "prefeeding",
    "feeding",
    "postfeeding"
]:
    os.makedirs(
        os.path.join(OUTPUT_DIR, cls),
        exist_ok=True
    )

for wav in os.listdir(SEGMENTS_DIR):

    if not wav.endswith(".wav"):
        continue

    try:
        ts = pd.to_datetime(
            wav.replace(".wav", ""),
            format="%Y-%m-%d_%H-%M-%S"
        )
    except:
        continue

    mask = (
        (intervals["start"] <= ts) &
        (ts < intervals["end"])
    )

    matches = intervals[mask]

    if len(matches) == 0:
        continue

    label = matches.iloc[0]["class"]

    shutil.copy2(
        os.path.join(SEGMENTS_DIR, wav),
        os.path.join(OUTPUT_DIR, label, wav)
    )

print("Done.")

Done.
